# Notebook 05: Expansion Target List & Board Memo
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Generate a prioritised expansion target list from the readiness scores. Calculate potential ARR impact. Write a board-level memo summarising the opportunity.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded")

In [ ]:
df = pd.read_csv('../data/processed/scored_accounts.csv')

print(f"Scored accounts: {len(df)}")
print(f"\nReadiness band distribution:")
print(df['readiness_band'].value_counts().sort_index())
print(f"\nAvg score by plan tier:")
print(df.groupby('plan_tier')['expansion_readiness_score']
      .mean().round(1).sort_values(ascending=False))

## 1. Expansion Target Prioritisation
Score bands define outreach priority tiers.

In [ ]:
# MRR uplift assumptions
MRR_UPLIFT = {
    'Basic': 941,       # Basic → Pro avg uplift from Notebook 02
    'Pro':   4286       # Pro → Enterprise avg uplift from Notebook 02
}

df['potential_mrr_uplift'] = df['plan_tier'].map(MRR_UPLIFT)
df['potential_arr_uplift'] = df['potential_mrr_uplift'] * 12

# Priority tiers
tier_map = {
    'Very High': 'Tier 1 — Immediate outreach',
    'High':      'Tier 2 — This quarter',
    'Medium':    'Tier 3 — Next quarter',
    'Low':       'Tier 4 — Monitor'
}
df['outreach_priority'] = df['readiness_band'].map(tier_map)

print("=== Expansion Target Summary by Priority Tier ===\n")
summary = (df.groupby('outreach_priority')
           .agg(
               accounts            =('account_id', 'count'),
               avg_score           =('expansion_readiness_score', 'mean'),
               total_mrr_uplift    =('potential_mrr_uplift', 'sum'),
               total_arr_uplift    =('potential_arr_uplift', 'sum')
           )
           .reset_index()
           .sort_values('avg_score', ascending=False))

summary['avg_score']        = summary['avg_score'].round(1)
summary['total_mrr_uplift'] = summary['total_mrr_uplift'].apply(
    lambda x: f"${x:,.0f}"
)
summary['total_arr_uplift'] = summary['total_arr_uplift'].apply(
    lambda x: f"${x:,.0f}"
)
print(summary.to_string(index=False))

In [ ]:
tier_colors = {
    'Tier 1 — Immediate outreach': '#27ae60',
    'Tier 2 — This quarter':       '#2ecc71',
    'Tier 3 — Next quarter':       '#e67e22',
    'Tier 4 — Monitor':            '#e74c3c'
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Accounts by Priority Tier',
        'Potential ARR by Priority Tier'
    ]
)

tier_counts = (df.groupby('outreach_priority')
               .agg(
                   accounts        =('account_id', 'count'),
                   total_arr       =('potential_arr_uplift', 'sum')
               )
               .reset_index()
               .sort_values('total_arr', ascending=False))

colors = [tier_colors[t] for t in tier_counts['outreach_priority']]

fig.add_trace(go.Bar(
    x=tier_counts['outreach_priority'],
    y=tier_counts['accounts'],
    marker_color=colors,
    text=tier_counts['accounts'],
    textposition='outside',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=tier_counts['outreach_priority'],
    y=tier_counts['total_arr'],
    marker_color=colors,
    text=[f"${x/1e6:.1f}M" for x in tier_counts['total_arr']],
    textposition='outside',
    showlegend=False
), row=1, col=2)

max_arr = tier_counts['total_arr'].max()
fig.update_yaxes(range=[0, tier_counts['accounts'].max() * 1.25],
                 row=1, col=1)
fig.update_yaxes(range=[0, max_arr * 1.25], row=1, col=2)

fig.update_layout(
    height=450,
    font=dict(size=12),
    title='Expansion Pipeline by Priority Tier'
)
fig.show()

## 2. Top 20 Expansion Targets
Tier 1 and Tier 2 accounts ranked by readiness score.

In [ ]:
top_targets = df[
    df['readiness_band'].isin(['Very High', 'High'])
].sort_values('expansion_readiness_score', ascending=False).head(20)

display_cols = [
    'account_name', 'industry', 'plan_tier',
    'total_mrr', 'total_seats',
    'total_usage_count', 'unique_features_used',
    'expansion_readiness_score', 'outreach_priority'
]

top_display = top_targets[display_cols].copy()
top_display['total_mrr'] = top_display['total_mrr'].apply(
    lambda x: f"${x:,.0f}"
)
top_display.columns = [
    'Account', 'Industry', 'Plan',
    'Current MRR', 'Seats',
    'Usage Count', 'Features Used',
    'Readiness Score', 'Priority'
]

print("=== Top 20 Expansion Targets ===\n")
print(top_display.to_string(index=False))

## 3. Expansion ARR Opportunity Summary

In [ ]:
tier1 = df[df['readiness_band'] == 'Very High']
tier2 = df[df['readiness_band'] == 'High']
tier12 = df[df['readiness_band'].isin(['Very High', 'High'])]

total_addressable_arr = df['potential_arr_uplift'].sum()
tier12_arr            = tier12['potential_arr_uplift'].sum()
tier1_arr             = tier1['potential_arr_uplift'].sum()

print("=== Expansion ARR Opportunity ===\n")
print(f"Total addressable expansion ARR:     ${total_addressable_arr:>12,.0f}")
print(f"Tier 1 + 2 (high confidence):        ${tier12_arr:>12,.0f}")
print(f"Tier 1 only (immediate):             ${tier1_arr:>12,.0f}")

print(f"\nAt 15% quarterly conversion:")
print(f"  Tier 1 + 2 target:  "
      f"${tier12_arr * 0.15:,.0f}/mo  "
      f"(${tier12_arr * 0.15 * 12:,.0f} ARR)")

print(f"\nBy plan tier:")
opp = (df.groupby('plan_tier')
       .agg(
           accounts            =('account_id', 'count'),
           avg_score           =('expansion_readiness_score', 'mean'),
           total_arr_potential =('potential_arr_uplift', 'sum')
       )
       .sort_values('total_arr_potential', ascending=False))
opp['avg_score']           = opp['avg_score'].round(1)
opp['total_arr_potential'] = opp['total_arr_potential'].apply(
    lambda x: f"${x:,.0f}"
)
print(opp.to_string())

In [ ]:
target_list = df.sort_values(
    'expansion_readiness_score', ascending=False
)[[
    'account_id', 'account_name', 'industry', 'country',
    'plan_tier', 'total_mrr', 'total_seats',
    'total_usage_count', 'unique_features_used',
    'avg_satisfaction', 'expansion_readiness_score',
    'readiness_band', 'outreach_priority',
    'potential_mrr_uplift', 'potential_arr_uplift'
]]

target_list.to_csv(
    '../data/processed/expansion_target_list.csv', index=False
)
print(f"Saved expansion_target_list.csv — {len(target_list)} accounts")

## 4. Board Memo

---

**Situation**

Net Revenue Retention stands at 96.7% — below the 100% threshold that signals a self-sustaining growth engine. Contraction MRR of $459K from downgrades is offsetting $1.26M in expansion gains. Without intervention, RavenStack must rely on new logo acquisition to grow, which is 5-7x more expensive than expanding existing accounts.

**Opportunity**

270 active Basic and Pro accounts represent $8.6M in addressable expansion ARR. Our expansion model identifies 70 high-confidence targets (Tier 1 and Tier 2) with the strongest upgrade signals — deep product engagement, large seat counts, and high MRR investment.

The single strongest predictor of expansion readiness is time spent in the product. Accounts in the top quartile of usage duration are upgrading at nearly double the rate of low-engagement accounts.

**Recommendation**

Three actions for Q1:

1. **Run two parallel expansion motions.**
   Basic accounts score higher on expansion readiness (56.9 vs 43.1) but Pro → Enterprise delivers 4.5x more ARR per conversion ($4,286
   vs $941/mo uplift). Dedicate senior CSMs to the 26 Tier 1 accounts regardless of plan — these are the highest-confidence opportunities.
   Use scaled digital outreach for the 114 Tier 2 accounts.

2. **Launch a usage-based expansion playbook.** Trigger automated CSM outreach when an account crosses the 90th percentile of usage duration. These accounts are hitting product limits and are the highest-conversion expansion opportunity.

3. **Address contraction before chasing expansion.** $459K in monthly contraction MRR is the silent revenue leak. A downgrade early-warning system — flagging accounts with declining usage — would recover more revenue than new upsells at current conversion rates.

**Expected outcome:** At 15% quarterly conversion of Tier 1 and Tier 2 targets, incremental ARR of $1.2M within two quarters. Combined with a 20% reduction in contraction MRR, NRR moves from 96.7% to approximately 103% — crossing the critical 100% threshold.

## 5. Key Findings

**270 active accounts represent $8.6M addressable expansion ARR:**
Pro → Enterprise dominates at $7.1M (83% of total) vs $1.5M
for Basic → Pro. However Basic accounts score higher on expansion
readiness (56.9 vs 43.1) — they are hitting product limits faster.
Two parallel expansion motions are needed: volume (Basic) and
value (Pro).

**26 Tier 1 accounts identified for immediate outreach:**
These accounts score 76-100 on expansion readiness.
Combined Tier 1 + Tier 2 (140 accounts) represent $3.5M in
addressable ARR. At 15% quarterly conversion: $520K/mo incremental
MRR or $6.2M ARR annually.

**DevTools and HealthTech dominate the top targets:**
5 of the top 10 accounts are DevTools industry.
HealthTech appears 4 times in the top 20.
Consistent with Notebook 02 — these are the highest-yield
expansion segments at 10.6% and 12.7% upgrade rates respectively.

**Usage duration is the #1 expansion signal:**
total_usage_duration appears in 9 of the top 10 SHAP explanations.
Accounts spending the most time in the product are most
expansion-ready. The multicollinearity between usage_duration
and usage_count means they appear with opposite signs in SHAP —
duration is the cleaner single signal.

**SHAP reasons make every score actionable:**
Each account has a top 3 SHAP explanation saved in
scored_accounts.csv. CSM teams open every expansion call
knowing exactly which signals drove the score.
These SHAP reasons can be passed directly to an AI system
to generate personalised outreach messaging at scale.

**NRR at 96.7% — contraction is the silent revenue leak:**
$459K monthly contraction MRR offsets expansion gains.
At current rates RavenStack needs new logo acquisition
just to stay flat. A downgrade early-warning system running
in parallel with expansion outreach would move NRR above
the critical 100% threshold faster than upsells alone.